In [2]:
!pip install astroquery

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 49.7 MB/s eta 0:00:00


In [3]:
"""
WILL Relational Orbital Mechanics (R.O.M.)
Official Validation Script: Continuous 3-Body Differential Phase Integration
Target: Juno Earth Flyby (Sept 29 - Oct 19, 2013)

Epistemological Premise:
This script demonstrates that N-body orbital mechanics can be calculated purely
through Relational Geometry (R.O.M.), without Newtonian mass (kg) or the
gravitational constant (G). The values of R_S_EARTH_M and R_S_SUN_M are
calculated without need of M and G as showed in the document.
It uses only structural potential capacities (kappa^2),
their spatial gradients (Gamma), and the local kinetic buffer (2*beta^2).
"""

import numpy as np
from astroquery.jplhorizons import Horizons
from astropy.time import Time

# =====================================================================
# I. STRICT SI CONSTANTS & ARCHITECTURAL PARAMETERS
# =====================================================================
C_M_S = 299792458.0           # Speed of light [m/s]
AU_M = 149597870700.0         # Astronomical Unit [m]
SEC_PER_DAY = 86400.0         # Seconds in a day [s]

# R.O.M. Fundamental Capacities (Schwarzschild-equivalent structural depths)
R_S_EARTH_M = 0.008870056     # Earth topological capacity [m]
R_S_SUN_M = 2953.25008        # Sun topological capacity [m]

def fetch_jpl_trajectory(target_id, center_id, start, stop, step):
    """
    Fetches raw ephemeris data and enforces strict conversion to meters and meters/second.
    """
    obj = Horizons(id=target_id, location=center_id, epochs={'start': start, 'stop': stop, 'step': step})
    vec = obj.vectors()

    jd = vec['datetime_jd']
    pos_m = np.column_stack((vec['x'], vec['y'], vec['z'])) * AU_M
    vel_ms = np.column_stack((vec['vx'], vec['vy'], vec['vz'])) * (AU_M / SEC_PER_DAY)

    return jd, pos_m, vel_ms

def calculate_angle(v1, v2):
    """Returns the scalar angle in radians between two 3D vectors."""
    unit_v1 = v1 / np.linalg.norm(v1)
    unit_v2 = v2 / np.linalg.norm(v2)
    return np.arccos(np.clip(np.dot(unit_v1, unit_v2), -1.0, 1.0))

def run_rom_validation():
    # We use the full 20-day window of the Juno flyby with a 1-minute integration step
    # to eliminate Euler numerical artifacts and capture the full macro-gradient.
    t_start = '2013-09-29 00:00:00'
    t_stop = '2013-10-19 00:00:00'
    t_step = '1m'

    print(f"Fetching {t_step} JPL ephemeris for R.O.M. Validation...\n")

    try:
        # 1. Probe w.r.t Earth (Local Geocentric Frame)
        jd, pos_PE, vel_PE = fetch_jpl_trajectory('-61', '@399', t_start, t_stop, t_step)
        # 2. Probe w.r.t Sun (Heliocentric Macro-Frame)
        _, pos_PS, _ = fetch_jpl_trajectory('-61', '@10', t_start, t_stop, t_step)
        # 3. Earth w.r.t Sun (Carrier Macro-Frame)
        _, pos_ES, _ = fetch_jpl_trajectory('399', '@10', t_start, t_stop, t_step)
    except Exception as e:
        print(f"API Error: {e}")
        return

    # =====================================================================
    # II. GROUND TRUTH & STATIC 2-BODY BASELINE
    # =====================================================================

    # Identify Perigee (Closest approach) to establish the invariant orbital plane
    idx_p = np.argmin(np.linalg.norm(pos_PE, axis=1))
    n_vec = np.cross(pos_PE[idx_p], vel_PE[idx_p])
    n_hat = n_vec / np.linalg.norm(n_vec)

    # 1. JPL Ground Truth: Total observed geocentric deflection angle
    v_in_obs = vel_PE[0]
    v_out_obs = vel_PE[-1]
    phi_obs_deg = np.degrees(calculate_angle(v_in_obs, v_out_obs))

    # 2. R.O.M. Static Baseline: The idealized 2-body topological defect
    # e = (2 * beta_p^2 / kappa_p^2) - 1
    r_p = np.linalg.norm(pos_PE[idx_p])
    beta_loc_in = np.linalg.norm(v_in_obs) / C_M_S
    kappa_pE_sq = R_S_EARTH_M / r_p
    e_base = (2.0 * ((beta_loc_in**2) + kappa_pE_sq) / kappa_pE_sq) - 1.0
    phi_base_deg = np.degrees(2.0 * np.arcsin(1.0 / e_base))

    # =====================================================================
    # III. CONTINUOUS 3-BODY DIFFERENTIAL INTEGRATION
    # =====================================================================
    # Instead of treating the flyby as a static bubble, we continuously
    # integrate the structural deformation of the trajectory.

    total_dphi_E_rad = 0.0
    total_dphi_S_rad = 0.0

    # Step through every minute of the 20-day trajectory
    for i in range(len(jd) - 1):
        dt_sec = (jd[i+1] - jd[i]) * SEC_PER_DAY

        # Current Kinematic State
        v_vec = vel_PE[i]
        v_mag = np.linalg.norm(v_vec)
        v_hat = v_vec / v_mag

        beta2 = (v_mag / C_M_S)**2
        ds = v_mag * dt_sec # Differential path length

        # -----------------------------------------------------
        # A. LOCAL DEFECT: Earth's Structural Gradient
        # Gamma = d(kappa^2)/dr = R_S / r^2
        # -----------------------------------------------------
        r_E_vec = -pos_PE[i] # Vector from Probe TO Earth
        r_E = np.linalg.norm(r_E_vec)
        u_E = r_E_vec / r_E

        Gamma_E = (R_S_EARTH_M / r_E) / r_E
        G_E_vec = Gamma_E * u_E

        # -----------------------------------------------------
        # B. MACRO-PERTURBATION: Sun's Relational Tidal Gradient
        # Because the coordinate center (Probe) is falling towards the Sun
        # alongside the Earth, the true curvature is generated by the
        # DIFFERENCE in the Sun's structural pull on the two bodies.
        # -----------------------------------------------------
        # Sun's pull on Probe
        r_PS_vec = -pos_PS[i]
        r_PS = np.linalg.norm(r_PS_vec)
        u_PS = r_PS_vec / r_PS
        Gamma_S_P = (R_S_SUN_M / r_PS) / r_PS
        G_S_P_vec = Gamma_S_P * u_PS

        # Sun's pull on Earth
        r_ES_vec = -pos_ES[i]
        r_ES = np.linalg.norm(r_ES_vec)
        u_ES = r_ES_vec / r_ES
        Gamma_S_E = (R_S_SUN_M / r_ES) / r_ES
        G_S_E_vec = Gamma_S_E * u_ES

        # The Relational Tidal Gradient
        G_S_tidal_vec = G_S_P_vec - G_S_E_vec

        # -----------------------------------------------------
        # C. KINEMATIC PROJECTION & PHASE UPDATE
        # Curvature is generated strictly by the lateral projection of
        # the gradient (cross product with velocity), restricted to the orbital plane.
        # -----------------------------------------------------
        lat_G_E = np.dot(np.cross(v_hat, G_E_vec), n_hat)
        lat_G_S = np.dot(np.cross(v_hat, G_S_tidal_vec), n_hat)

        # CORE R.O.M. DIFFERENTIAL EQUATION:
        # d(phi) = (Gamma_lateral / 2*beta^2) * ds
        # The deformation is inversely proportional to the 2*beta^2 kinetic buffer.
        dphi_E = (lat_G_E / (2.0 * beta2)) * ds
        dphi_S = (lat_G_S / (2.0 * beta2)) * ds

        total_dphi_E_rad += dphi_E
        total_dphi_S_rad += dphi_S

    # =====================================================================
    # IV. SYNTHESIS & OUTPUT
    # =====================================================================

    integrated_E_deg = np.degrees(total_dphi_E_rad)
    integrated_S_deg = np.degrees(total_dphi_S_rad)
    integrated_Total_deg = integrated_E_deg + integrated_S_deg

    anomaly_deg = phi_obs_deg - integrated_Total_deg

    print("==========================================================")
    print("      WILL RELATIONAL ORBITAL MECHANICS (R.O.M.)")
    print("      Continuous 3-Body Phase Integration Report")
    print("==========================================================")
    print(f"JPL Observed Geocentric Deflection:  {phi_obs_deg:.4f} deg")
    print(f"Static 2-Body Baseline Expectation:  {phi_base_deg:.4f} deg")
    print("----------------------------------------------------------")
    print(f"R.O.M. Integrated Earth Defect:      {integrated_E_deg:.4f} deg")
    print(f"R.O.M. Integrated Sun Tidal Shift:   {integrated_S_deg:.4f} deg")
    print("----------------------------------------------------------")
    print(f"Total Integrated R.O.M. Deflection:  {integrated_Total_deg:.4f} deg")
    print(f"Angle Anomaly (Observed - ROM):      {anomaly_deg:.4f} deg")
    print("==========================================================")
    print(f"Absolute Methodological Accuracy:    {100.0 - abs(anomaly_deg/phi_obs_deg)*100.0:.4f} %")

if __name__ == '__main__':
    run_rom_validation()

Fetching 1m JPL ephemeris for R.O.M. Validation...

      WILL RELATIONAL ORBITAL MECHANICS (R.O.M.)
      Continuous 3-Body Phase Integration Report
JPL Observed Geocentric Deflection:  40.8877 deg
Static 2-Body Baseline Expectation:  39.4337 deg
----------------------------------------------------------
R.O.M. Integrated Earth Defect:      40.6772 deg
R.O.M. Integrated Sun Tidal Shift:   0.1823 deg
----------------------------------------------------------
Total Integrated R.O.M. Deflection:  40.8595 deg
Angle Anomaly (Observed - ROM):      0.0282 deg
Absolute Methodological Accuracy:    99.9311 %


In [4]:
def validate_method_b(pos_PE, vel_PE):
    """
    Earth's Rs calculation implements Method B: Two-Point Schwarzschild Scale
    Rs = [ (r1 * r2) / (r2 - r1) ] * (beta1^2 - beta2^2)
    """
    # Point 1: Entry into the Earth's local system (index 0)
    r1 = np.linalg.norm(pos_PE[0])
    beta1_sq = (np.linalg.norm(vel_PE[0]) / C_M_S)**2

    # Point 2: Perigee (closest approach)
    idx_p = np.argmin(np.linalg.norm(pos_PE, axis=1))
    r2 = np.linalg.norm(pos_PE[idx_p])
    beta2_sq = (np.linalg.norm(vel_PE[idx_p]) / C_M_S)**2

    # Calculation of Rs using purely geometric/kinematic deltas
    # Note: We take absolute difference to handle the vector directionality
    delta_beta_sq = abs(beta1_sq - beta2_sq)
    delta_r_inv = abs((r2 - r1) / (r1 * r2))

    derived_rs = delta_beta_sq / delta_r_inv

    print("--- Method B: Two-Point Scale Validation ---")
    print(f"Point 1 (Entry) r: {r1/1000:.2f} km, beta^2: {beta1_sq:.2e}")
    print(f"Point 2 (Perigee) r: {r2/1000:.2f} km, beta^2: {beta2_sq:.2e}")
    print(f"Target Earth Rs:     {R_S_EARTH_M:.9f} m")
    print(f"Derived R.O.M. Rs:   {derived_rs:.9f} m")
    print(f"Calculation Error:   {abs(derived_rs - R_S_EARTH_M):.2e} m")

# Assuming run_rom_validation is still in memory, we can use the data from the previous run
# For this demonstration, we'll re-fetch or use variables if they were globalized.
# Since they were local to run_rom_validation, let's extract the logic:

t_start, t_stop, t_step = '2013-09-29 00:00:00', '2013-10-19 00:00:00', '1m'
_, pos_PE_val, vel_PE_val = fetch_jpl_trajectory('-61', '@399', t_start, t_stop, t_step)
validate_method_b(pos_PE_val, vel_PE_val)

--- Method B: Two-Point Scale Validation ---
Point 1 (Entry) r: 9791814.67 km, beta^2: 1.25e-09
Point 2 (Perigee) r: 6941.82 km, beta^2: 2.48e-09
Target Earth Rs:     0.008870056 m
Derived R.O.M. Rs:   0.008502360 m
Calculation Error:   3.68e-04 m
